<a href="https://colab.research.google.com/github/gauravsinha12/AI-CLI/blob/main/Algo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
"""
01_feature_engineering.py
=========================
Institutional Apex Pipeline
"""

import sys
import polars as pl
import numpy as np

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
INPUT_PATH  = "/content/master_nifty100_5m_2024_2026.csv"
OUTPUT_PATH = "ml_ready_nifty100.csv"

DATE_FMT = "%Y-%m-%dT%H:%M:%S%.f"
FORECAST_BARS    = 12         # 1 Hour horizon
WILDER_ALPHA     = 1 / 14
VOL_WINDOW       = 20
ATR_TARGET_MULT  = 0.5
RELVOL_CLIP      = 10.0

# ---------------------------------------------------------------------------
# 1. Load & sort
# ---------------------------------------------------------------------------
print(f"[1/9] Loading {INPUT_PATH} ...")
try:
    # Explicitly enforce data types during ingestion to guarantee safety
    df = pl.read_csv(
        INPUT_PATH,
        has_header=False,
        new_columns=["date", "open", "high", "low", "close", "volume", "ticker_id"],
        schema_overrides={
            "open": pl.Float64,
            "high": pl.Float64,
            "low": pl.Float64,
            "close": pl.Float64,
            "volume": pl.Int64,
            "ticker_id": pl.String,
            "date": pl.String
        }
    )
except FileNotFoundError:
    sys.exit(f"ERROR: {INPUT_PATH} not found. Check your path.")
except Exception as e:
    print(f"Schema-enforced parse failed, attempting standard cast... Error: {e}")
    df = pl.read_csv(INPUT_PATH)
    # Fallback inline type casting
    df = df.with_columns([
        pl.col("open").cast(pl.Float64, strict=False),
        pl.col("high").cast(pl.Float64, strict=False),
        pl.col("low").cast(pl.Float64, strict=False),
        pl.col("close").cast(pl.Float64, strict=False),
        pl.col("volume").cast(pl.Int64, strict=False),
    ])

# Sort: ticker first, then time — CRITICAL for all rolling ops
df = df.sort(["ticker_id", "date"])

# ---------------------------------------------------------------------------
# 2. Parse datetime + session_date
# ---------------------------------------------------------------------------
print("[2/9] Parsing datetime ...")
df = (
    df
    .with_columns([
        pl.col("date").str.to_datetime(DATE_FMT, strict=False).alias("datetime")
    ])
    .with_columns([
        pl.col("datetime").dt.date().alias("session_date")
    ])
)

# ---------------------------------------------------------------------------
# 3. Bar counter, Previous Close, and Session Boundaries
# ---------------------------------------------------------------------------
print("[3/9] Computing Session Boundaries & ATR ...")
df = df.with_columns([
    pl.col("close").cum_count().over(["ticker_id", "session_date"]).alias("bar_of_day"),
    pl.col("close").shift(1).over("ticker_id").alias("prev_close"),
    pl.col("open").first().over(["ticker_id", "session_date"]).alias("session_open"),
    pl.col("high").cum_max().over(["ticker_id", "session_date"]).alias("session_high"),
    pl.col("low").cum_min().over(["ticker_id", "session_date"]).alias("session_low"),
])

df = df.with_columns([
    pl.max_horizontal([
        pl.col("high") - pl.col("low"),
        (pl.col("high") - pl.col("prev_close")).abs(),
        (pl.col("low")  - pl.col("prev_close")).abs(),
    ]).alias("true_range")
])

df = df.with_columns([
    pl.col("true_range").ewm_mean(alpha=WILDER_ALPHA, adjust=False, ignore_nulls=True).over("ticker_id").alias("atr_14"),
    pl.col("volume").shift(1).rolling_mean(window_size=VOL_WINDOW).over("ticker_id").alias("avg_vol_20_prev"),
    pl.col("close").ewm_mean(span=12, adjust=False, ignore_nulls=True).over("ticker_id").alias("ema_12"),
    pl.col("close").ewm_mean(span=26, adjust=False, ignore_nulls=True).over("ticker_id").alias("ema_26")
])

# ---------------------------------------------------------------------------
# 4. VWAP & Base Market Drivers
# ---------------------------------------------------------------------------
print("[4/9] Computing VWAP & Base Drivers ...")
df = df.with_columns([
    (((pl.col("high") + pl.col("low") + pl.col("close")) / 3) * pl.col("volume"))
      .cum_sum().over(["ticker_id", "session_date"]).alias("cum_pv"),
    pl.col("volume")
      .cum_sum().over(["ticker_id", "session_date"]).alias("cum_vol"),
])

vwap_expr = pl.col("cum_pv") / (pl.col("cum_vol") + 1e-10)
minutes = (pl.col("datetime").dt.hour() * 60 + pl.col("datetime").dt.minute() - 555)

# ---------------------------------------------------------------------------
# 5. The Institutional Feature Matrix
# ---------------------------------------------------------------------------
print("[5/9] Compiling Institutional Feature Matrix ...")

df = df.with_columns([
    # 1. Structural Price Action
    ((pl.col("close") - vwap_expr) / (pl.col("atr_14") + 1e-10)).alias("vwap_atr_dev"),
    ((pl.col("session_open") - pl.col("prev_close")) / (pl.col("atr_14") + 1e-10)).alias("overnight_gap"),
    ((pl.col("close") - pl.col("session_low")) / (pl.col("session_high") - pl.col("session_low") + 1e-10)).alias("range_position"),

    # 2. Normalized Momentum (MACD proxy mapped to ATR)
    ((pl.col("ema_12") - pl.col("ema_26")) / (pl.col("atr_14") + 1e-10)).alias("macd_atr"),

    # 3. Volume Intensity
    (pl.col("volume") / (pl.col("avg_vol_20_prev") + 1e-10)).clip(0.0, RELVOL_CLIP).alias("relative_volume"),

    # 4. Cyclical Time Encoding
    (minutes * (2 * np.pi / 375)).sin().alias("time_sin"),
    (minutes * (2 * np.pi / 375)).cos().alias("time_cos"),

    # Intermediate variable for trend
    ((pl.col("close") - pl.col("close").shift(6).over("ticker_id")) / (pl.col("atr_14") + 1e-10)).alias("stock_trend")
])

# ---------------------------------------------------------------------------
# 5.5. True Alpha (Beta-Neutrality)
# ---------------------------------------------------------------------------
print("[5.5/9] Computing True Alpha (Beta Neutrality) ...")
df = df.with_columns([
    pl.col("stock_trend").mean().over("datetime").alias("market_trend")
])

df = df.with_columns([
    # Idiosyncratic momentum: How fast is this moving relative to the market?
    (pl.col("stock_trend") - pl.col("market_trend")).alias("idiosyncratic_alpha")
])

# ---------------------------------------------------------------------------
# 6. Cross-sectional ranks
# ---------------------------------------------------------------------------
print("[6/9] Computing cross-sectional ranks ...")
CS_FEATURES = ["vwap_atr_dev", "range_position", "macd_atr", "relative_volume", "idiosyncratic_alpha"]

df = df.with_columns([
    (
        pl.col(feat).rank(method="average").over("datetime")
        / pl.col(feat).count().over("datetime")
    ).alias(f"{feat}_rank")
    for feat in CS_FEATURES
])

# ---------------------------------------------------------------------------
# 7. Execution Target (1-Hour Horizon)
# ---------------------------------------------------------------------------
print("[7/9] Building execution-realistic target (1-Hour Horizon) ...")
df = df.with_columns([
    pl.col("open").shift(-1).over(["ticker_id", "session_date"]).alias("next_open"),
    pl.col("close").shift(-FORECAST_BARS).over(["ticker_id", "session_date"]).alias("future_close"),
])

df = df.with_columns([
    (((pl.col("future_close") - pl.col("next_open")) / pl.col("next_open")) * 100.0).alias("future_return_pct"),
    ((pl.col("atr_14") / pl.col("close")) * 100.0 * ATR_TARGET_MULT).alias("dynamic_threshold"),
])

df = df.with_columns([
    pl.when(pl.col("future_return_pct") >  pl.col("dynamic_threshold")).then(pl.lit( 1))
      .when(pl.col("future_return_pct") < -pl.col("dynamic_threshold")).then(pl.lit(-1))
      .otherwise(pl.lit(0))
      .cast(pl.Int8)
      .alias("target_direction")
])

# ---------------------------------------------------------------------------
# 8. Final cleanup
# ---------------------------------------------------------------------------
print("[8/9] Cleaning up ...")
FINAL_COLS = [
    "datetime", "session_date", "ticker_id",
    "target_direction", "future_return_pct",
    "time_sin", "time_cos", "market_trend",
    "vwap_atr_dev", "overnight_gap", "range_position", "macd_atr", "relative_volume", "idiosyncratic_alpha",
    "vwap_atr_dev_rank", "range_position_rank", "macd_atr_rank", "relative_volume_rank", "idiosyncratic_alpha_rank"
]

final_df = df.filter(pl.col("bar_of_day") > 1).select(FINAL_COLS).drop_nulls()

n_rows  = final_df.height
classes = final_df["target_direction"].value_counts().sort("target_direction")

print(f"\n[9/9] Saved {n_rows:,} rows -> {OUTPUT_PATH}")
print(f"Target distribution:\n{classes}")
final_df.write_csv(OUTPUT_PATH)

[1/9] Loading /content/master_nifty100_5m_2024_2026.csv ...
Schema-enforced parse failed, attempting standard cast... Error: could not parse `open` as dtype `f64` at column 'column_2' (column number 2)

The current offset in the file is 5 bytes.

You might want to try:
- increasing `infer_schema_length` (e.g. `infer_schema_length=10000`),
- specifying correct dtype with the `schema_overrides` argument
- setting `ignore_errors` to `True`,
- adding `open` to the `null_values` list.

Original error: ```invalid primitive value found during CSV parsing```
[2/9] Parsing datetime ...
[3/9] Computing Session Boundaries & ATR ...
[4/9] Computing VWAP & Base Drivers ...
[5/9] Compiling Institutional Feature Matrix ...
[5.5/9] Computing True Alpha (Beta Neutrality) ...
[6/9] Computing cross-sectional ranks ...
[7/9] Building execution-realistic target (1-Hour Horizon) ...
[8/9] Cleaning up ...

[9/9] Saved 3,483,415 rows -> ml_ready_nifty100.csv
Target distribution:
shape: (3, 2)
┌───────────────

In [6]:
"""
02_train_lgbm.py
================
Institutional LightGBM Training Pipeline
"""

import json
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import polars as pl
from scipy.stats import spearmanr

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DATA_PATH      = "/content/ml_ready_nifty100.csv"
MODEL_PATH     = "universal_multiclass_model.txt"
ARTIFACTS_PATH = "universal_multiclass_artifacts.json"

RAW_FEATURES  = ["time_sin", "time_cos", "market_trend", "vwap_atr_dev", "overnight_gap", "range_position", "macd_atr", "relative_volume", "idiosyncratic_alpha"]
RANK_FEATURES = ["vwap_atr_dev_rank", "range_position_rank", "macd_atr_rank", "relative_volume_rank", "idiosyncratic_alpha_rank"]
FEATURES      = RAW_FEATURES + RANK_FEATURES

TARGET_RAW_COL = "target_direction"
TARGET_COL     = "ml_target"
DATE_COL       = "datetime"

TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
GAP_DAYS   = 5

# INSTITUTIONAL HYPERPARAMETERS
LGBM_PARAMS = dict(
    objective          = "multiclass",
    num_class          = 3,
    n_estimators       = 4000,       # High capacity
    learning_rate      = 0.01,       # Slow, meticulous learning
    max_depth          = 6,
    num_leaves         = 45,
    min_child_samples  = 100,        # Requires massive evidence to split
    subsample          = 0.7,        # Row bagging
    colsample_bytree   = 0.4,        # Feature bagging (FORCES it to stop relying purely on time)
    reg_alpha          = 0.05,       # L1
    reg_lambda         = 0.5,        # Heavy L2
    random_state       = 42,
    n_jobs             = -1,
    verbose            = -1,
)

EARLY_STOP_ROUNDS = 200
LOG_PERIOD        = 200
THRESHOLDS        = np.arange(0.40, 0.80, 0.05)
MIN_SIGNAL_COUNT  = 20

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def ic_test(X, y_raw, feature_names):
    print("\n--- Information Coefficient (Spearman) ---")
    for i, name in enumerate(feature_names):
        ic, pval = spearmanr(X[:, i], y_raw)
        flag = "  <<< LOW SIGNAL" if abs(ic) < 0.01 else ""
        print(f"  {name:30s}  IC={ic:+.4f}  p={pval:.3f}{flag}")

def threshold_sweep(y_prob, y_test, future_returns, side, class_idx, true_label, ret_sign):
    pred_class = np.argmax(y_prob, axis=1)
    print(f"\n--- {side.upper()} SIGNAL SWEEP ---")
    any_printed = False

    for t in THRESHOLDS:
        mask = (pred_class == class_idx) & (y_prob[:, class_idx] >= t)
        if mask.sum() < MIN_SIGNAL_COUNT: continue
        any_printed = True
        precision = np.mean(y_test[mask] == true_label)
        coverage  = mask.sum() / len(y_test)

        if future_returns is not None:
            signed_ret  = ret_sign * future_returns[mask]
            net_ret = signed_ret - 0.05  # Brutal Institutional Reality Deducted
            avg_edge    = net_ret.mean()
            edge_sharpe = avg_edge / (net_ret.std() + 1e-10)
            print(f"  t={t:.2f} | n={mask.sum():5d} | cov={coverage:.3f} | prec={precision:.3f} | netRet={avg_edge:+.4f}% | Sharpe={edge_sharpe:.3f}")
        else:
            print(f"  t={t:.2f} | n={mask.sum():5d} | cov={coverage:.3f} | prec={precision:.3f}")
    if not any_printed: print(f"  No threshold produced >= {MIN_SIGNAL_COUNT} signals.")

# ---------------------------------------------------------------------------
# Pipeline Execution
# ---------------------------------------------------------------------------
print(f"[1/7] Loading {DATA_PATH} ...")
df = pl.read_csv(DATA_PATH)

if df[DATE_COL].dtype == pl.Utf8:
    df = df.with_columns([pl.col(DATE_COL).str.to_datetime(strict=False).alias(DATE_COL)])

df = df.with_columns([(pl.col(TARGET_RAW_COL) + 1).cast(pl.Int32).alias(TARGET_COL)])
use_cols = FEATURES + [TARGET_COL, DATE_COL, "session_date", "future_return_pct"]
df = df.drop_nulls(use_cols).sort(DATE_COL)

session_dates = df.select("session_date").unique().sort("session_date").get_column("session_date").to_list()
n_days = len(session_dates)

train_end = int(n_days * TRAIN_FRAC)
val_end   = int(n_days * (TRAIN_FRAC + VAL_FRAC))

train_dates = session_dates[:train_end]
val_dates   = session_dates[train_end + GAP_DAYS : val_end + GAP_DAYS]
test_dates  = session_dates[val_end   + GAP_DAYS * 2 :]

train_df = df.filter(pl.col("session_date").is_in(train_dates))
val_df   = df.filter(pl.col("session_date").is_in(val_dates))
test_df  = df.filter(pl.col("session_date").is_in(test_dates))

print(f"  Train: {len(train_dates)} days | Val: {len(val_dates)} days | Test: {len(test_dates)} days")

X_train, y_train = train_df.select(FEATURES).to_numpy().astype(np.float32), train_df.get_column(TARGET_COL).to_numpy()
X_val, y_val     = val_df.select(FEATURES).to_numpy().astype(np.float32), val_df.get_column(TARGET_COL).to_numpy()
X_test, y_test   = test_df.select(FEATURES).to_numpy().astype(np.float32), test_df.get_column(TARGET_COL).to_numpy()
future_returns   = test_df.get_column("future_return_pct").to_numpy()
y_raw_train      = train_df.get_column("future_return_pct").to_numpy()

ic_test(X_train, y_raw_train, FEATURES)

print("\n[4/7] Training LightGBM ...")
clf = lgb.LGBMClassifier(**LGBM_PARAMS)
clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="multi_logloss", callbacks=[lgb.early_stopping(EARLY_STOP_ROUNDS, first_metric_only=True, verbose=True), lgb.log_evaluation(LOG_PERIOD)])

best_iter = clf.best_iteration_
print(f"\n[5/7] Feature importance (gain) ...")
importances = clf.booster_.feature_importance(importance_type="gain")
feat_imp = sorted(zip(FEATURES, importances), key=lambda x: x[1], reverse=True)
for name, imp in feat_imp: print(f"  {name:30s} {'█' * int(imp / max(importances) * 30)} {imp:.1f}")

print("\n[6/7] Evaluating on test set ...")
y_prob = clf.predict_proba(X_test, num_iteration=best_iter)

threshold_sweep(y_prob, y_test, future_returns, "long", 2, 2, 1.0)
threshold_sweep(y_prob, y_test, future_returns, "short", 0, 0, -1.0)

print(f"\n[7/7] Saving to {MODEL_PATH} ...")
clf.booster_.save_model(MODEL_PATH)

# Save artifacts JSON
artifacts = {
    "model_path": MODEL_PATH,
    "features": FEATURES,
    "target_mapping": {"-1": 0, "0": 1, "1": 2},
    "lgbm_params": {k: str(v) for k, v in LGBM_PARAMS.items()},
    "best_iteration": int(best_iter) if best_iter is not None else None,
}
Path(ARTIFACTS_PATH).write_text(json.dumps(artifacts, indent=2, default=str), encoding="utf-8")
print(f"[7/7] Saving artifacts to {ARTIFACTS_PATH} ...")

[1/7] Loading /content/ml_ready_nifty100.csv ...
  Train: 393 days | Val: 84 days | Test: 75 days

--- Information Coefficient (Spearman) ---
  time_sin                        IC=+0.0005  p=0.451  <<< LOW SIGNAL
  time_cos                        IC=+0.0026  p=0.000  <<< LOW SIGNAL
  market_trend                    IC=+0.0045  p=0.000  <<< LOW SIGNAL
  vwap_atr_dev                    IC=+0.0130  p=0.000
  overnight_gap                   IC=-0.0130  p=0.000
  range_position                  IC=+0.0154  p=0.000
  macd_atr                        IC=+0.0126  p=0.000
  relative_volume                 IC=+0.0004  p=0.511  <<< LOW SIGNAL
  idiosyncratic_alpha             IC=-0.0058  p=0.000  <<< LOW SIGNAL
  vwap_atr_dev_rank               IC=-0.0017  p=0.007  <<< LOW SIGNAL
  range_position_rank             IC=+0.0006  p=0.353  <<< LOW SIGNAL
  macd_atr_rank                   IC=-0.0024  p=0.000  <<< LOW SIGNAL
  relative_volume_rank            IC=+0.0036  p=0.000  <<< LOW SIGNAL
  idiosyncra

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



--- LONG SIGNAL SWEEP ---
  t=0.40 | n=42005 | cov=0.088 | prec=0.415 | netRet=-0.0204% | Sharpe=-0.031
  t=0.45 | n= 5379 | cov=0.011 | prec=0.414 | netRet=-0.0166% | Sharpe=-0.019
  t=0.50 | n=  746 | cov=0.002 | prec=0.629 | netRet=+0.4945% | Sharpe=0.417
  t=0.55 | n=  284 | cov=0.001 | prec=0.729 | netRet=+0.9808% | Sharpe=0.693
  t=0.60 | n=  188 | cov=0.000 | prec=0.723 | netRet=+0.9500% | Sharpe=0.661
  t=0.65 | n=  129 | cov=0.000 | prec=0.775 | netRet=+1.1286% | Sharpe=0.723
  t=0.70 | n=   57 | cov=0.000 | prec=0.860 | netRet=+1.4976% | Sharpe=0.855

--- SHORT SIGNAL SWEEP ---
  t=0.40 | n=127701 | cov=0.268 | prec=0.409 | netRet=-0.0449% | Sharpe=-0.071
  t=0.45 | n= 3809 | cov=0.008 | prec=0.440 | netRet=-0.0123% | Sharpe=-0.013

[7/7] Saving to universal_multiclass_model.txt ...
[7/7] Saving artifacts to universal_multiclass_artifacts.json ...
